# Population - Bronze Ingestion

### Imports and Parameters

In [ ]:
%pip install openpyxl

In [ ]:
import json
import os
import re
from datetime import UTC, date, datetime

from openpyxl import load_workbook
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

In [ ]:
dbutils.widgets.text("catalog", "public_health", "Catalog (your own)")
dbutils.widgets.text(
    "landing_path",
    "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/population",
    "Population landing folder",
)
dbutils.widgets.dropdown("reload_all", "false", ["false", "true"], "Reload loaded batches")

CATALOG = dbutils.widgets.get("catalog")
LANDING_PATH = dbutils.widgets.get("landing_path")
RELOAD_ALL = dbutils.widgets.get("reload_all") == "true"

TABLE = f"{CATALOG}.bronze.population_bronze"
N_COLS = 14  # c0..c13; anything wider goes to _rescued_data
FILE_PATTERN = re.compile(r"^population_(?P<year>\d{4})_asof_(?P<asof>\d{8})\.xlsx$")


### Helper Functions

In [ ]:
def parse_as_of_date(file_name: str) -> date:
    """population_2024_asof_20250717.xlsx -> date(2025, 7, 17). Raises ValueError if bad name."""
    match = FILE_PATTERN.match(file_name)
    if match is None:
        raise ValueError(f"{file_name} does not match population_<year>_asof_<YYYYMMDD>.xlsx")
    return datetime.strptime(match.group("asof"), "%Y%m%d").date()


def cell_to_str(value) -> str | None:
    """Cell -> string without changing its meaning. Blank/whitespace-only -> None."""
    if value is None:
        return None
    if isinstance(value, float) and value.is_integer():
        return str(int(value))  # 1234567.0 -> "1234567"
    if isinstance(value, datetime | date):
        return value.isoformat()
    text = str(value)
    return text if text.strip() else None


def read_rows(path: str, n_cols: int):
    """Yield (sheet, excel_row_num, [n_cols cells], rescued_json) for every non-empty row."""
    wb = load_workbook(path, read_only=True, data_only=True)
    try:
        for ws in wb.worksheets:
            ws.reset_dimensions()  # some files store a wrong sheet size
            for row_num, row in enumerate(ws.iter_rows(values_only=True), start=1):
                values = [cell_to_str(v) for v in row]
                if all(v is None for v in values):
                    continue  # skip blank rows, but row_num still counts them
                cells = values[:n_cols] + [None] * (n_cols - len(values))
                extra = {f"c{i}": v for i, v in enumerate(values[n_cols:], start=n_cols) if v}
                yield ws.title, row_num, cells, (json.dumps(extra) if extra else None)
    finally:
        wb.close()

### Schema

In [ ]:
SCHEMA = StructType(
    [StructField(f"c{i}", StringType()) for i in range(N_COLS)]
    + [
        StructField("_rescued_data", StringType()),
        StructField("_source_file", StringType()),
        StructField("_source_sheet", StringType()),
        StructField("_source_row_num", IntegerType()),
        StructField("_as_of_date", DateType()),
        StructField("_ingested_at", TimestampType()),
    ]
)

In [ ]:
valid, ignored = {}, []
for name in sorted(os.listdir(LANDING_PATH)):
    try:
        valid[name] = parse_as_of_date(name)
    except ValueError:
        ignored.append(name)

if ignored:
    print(f"Ignored (name does not follow the convention): {ignored}")

already_loaded = set()
if spark.catalog.tableExists(TABLE):
    already_loaded = {r[0] for r in spark.table(TABLE).select("_source_file").distinct().collect()}

to_load = sorted(
    (n for n in valid if RELOAD_ALL or n not in already_loaded),
    key=lambda n: (valid[n], n),  # oldest as-of first
)
print(f"valid: {len(valid)} | already loaded: {len(already_loaded)} | to load: {to_load}")


In [ ]:
for name in to_load:
    as_of_date = valid[name]
    ingested_at = datetime.now(UTC)  # one timestamp per batch

    rows = [
        (*cells, rescued, name, sheet, row_num, as_of_date, ingested_at)
        for sheet, row_num, cells, rescued in read_rows(f"{LANDING_PATH}/{name}", N_COLS)
    ]
    # Tuple order MUST match SCHEMA: c0..c13, _rescued_data, _source_file, _source_sheet,
    # _source_row_num, _as_of_date, _ingested_at
    df = spark.createDataFrame(rows, schema=SCHEMA)

    writer = df.write.format("delta")
    if spark.catalog.tableExists(TABLE):
        # Replace only this file's rows -> re-running never duplicates
        safe_name = name.replace("'", "''")
        writer = writer.mode("overwrite").option("replaceWhere", f"_source_file = '{safe_name}'")
    else:
        writer = writer.mode("append")  # first batch creates the table
    writer.saveAsTable(TABLE)

    print(f"Loaded {name}: {len(rows)} rows, as_of {as_of_date}")

if not to_load:
    print("Nothing to load.")

In [ ]:
# Rows per batch/sheet. Compare with the non-empty row count in Excel.
display(
    spark.table(TABLE)
    .groupBy("_source_file", "_source_sheet", "_as_of_date")
    .agg(F.count("*").alias("rows"), F.max("_ingested_at").alias("last_ingested_at"))
    .orderBy("_as_of_date", "_source_file", "_source_sheet")
)

In [ ]:
# Eyeball the first rows: where is the real header row?
display(spark.table(TABLE).orderBy("_source_file", "_source_sheet", "_source_row_num").limit(15))


In [ ]:
# Must be empty. If not, a file is wider than N_COLS.
display(spark.table(TABLE).where("_rescued_data IS NOT NULL").limit(20))